# Modeling Volatility Risk Across Event Categories in Prediction Markets

**Research question:** Can market category predict future probability volatility in prediction markets, even after controlling for liquidity, volume, and time-to-expiry?

This notebook uses Polymarket data pulled from the live Gamma + CLOB APIs, with daily probability histories and market-level microstructure controls.

## Setup

This notebook assumes it is being run from either the project root or the `notebooks/` directory.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
data_dir = project_root / "data"

analytical = pd.read_csv(data_dir / "analytical_dataset.csv", parse_dates=["date"])
scoring = pd.read_csv(data_dir / "scoring_dataset.csv", parse_dates=["date"])
coefficients = pd.read_csv(data_dir / "ols_coefficients.csv")
rankings = pd.read_csv(data_dir / "category_rankings.csv")
importances = pd.read_csv(data_dir / "rf_feature_importances.csv")
volatility_over_time = pd.read_csv(data_dir / "volatility_over_time.csv", parse_dates=["date"])
with open(data_dir / "model_metrics.json", "r", encoding="utf-8") as handle:
    metrics = json.load(handle)

analytical.head()

## Data Coverage

In [ ]:
category_summary = (
    scoring.groupby("category", as_index=False)
    .agg(
        markets=("market_id", "nunique"),
        observations=("market_id", "size"),
        median_volatility=("rolling_volatility_7d", "median"),
        mean_volatility=("rolling_volatility_7d", "mean"),
    )
    .sort_values("markets", ascending=False)
)
category_summary["underrepresented_lt_20"] = category_summary["markets"] < 20
display(category_summary)

print("Training rows:", metrics["n_rows"])
print("Markets:", metrics["n_markets"])
print("OLS adjusted R²:", round(metrics["ols_adj_r2"], 3))
print("Random forest mean CV R²:", round(metrics["rf_mean_r2"], 3))

## Volatility by Category

In [ ]:
order = (
    scoring.groupby("category")["rolling_volatility_7d"]
    .median()
    .sort_values(ascending=False)
    .index
)

fig = px.box(
    scoring,
    x="category",
    y="rolling_volatility_7d",
    category_orders={"category": order.tolist()},
    points=False,
    title="Distribution of 7-day rolling volatility by category",
)
fig.update_layout(xaxis_title="", yaxis_title="7d rolling volatility")
fig.show()

## Volatility Over Time

In [ ]:
fig = px.line(
    volatility_over_time,
    x="date",
    y="avg_rolling_volatility_7d",
    color="category",
    title="Average category-level volatility over time",
)
fig.update_layout(xaxis_title="", yaxis_title="Average 7d rolling volatility")
fig.show()

## Correlation Matrix

In [ ]:
corr_columns = [
    "rolling_volatility_7d",
    "future_volatility_7d",
    "log_volume",
    "bid_ask_spread",
    "time_to_expiry_days",
    "prob_level",
    "recent_volatility_7d",
]
corr = analytical[corr_columns].corr(numeric_only=True)
heatmap = px.imshow(
    corr,
    text_auto=".2f",
    color_continuous_scale="Blues",
    title="Feature correlation matrix",
)
heatmap.show()

## Volatility vs Time to Expiry

In [ ]:
sample = analytical.sample(min(len(analytical), 6000), random_state=42)
fig = px.scatter(
    sample,
    x="time_to_expiry_days",
    y="rolling_volatility_7d",
    color="category",
    opacity=0.35,
    trendline="lowess",
    title="Does realized volatility rise near expiry?",
)
fig.update_layout(xaxis_title="Days to expiry", yaxis_title="7d rolling volatility")
fig.show()

## Volatility vs Probability Level

In [ ]:
sample = analytical.sample(min(len(analytical), 6000), random_state=7)
fig = px.scatter(
    sample,
    x="prob_level",
    y="rolling_volatility_7d",
    color="category",
    opacity=0.35,
    trendline="lowess",
    title="Volatility vs implied probability level",
)
fig.update_layout(xaxis_title="Implied probability", yaxis_title="7d rolling volatility")
fig.show()

## OLS Regression Results

In [ ]:
display(coefficients.sort_values("p_value"))

significant_categories = coefficients[
    coefficients["term"].str.contains("C(category", regex=False, na=False)
    & (coefficients["p_value"] < 0.05)
][["term", "coef", "p_value"]]
significant_categories

## Random Forest Feature Importance

In [ ]:
fig = px.bar(
    importances.sort_values("importance", ascending=True),
    x="importance",
    y="feature_group",
    orientation="h",
    title="Aggregated random forest feature importances",
    color="importance",
    color_continuous_scale="Blues",
)
fig.update_layout(coloraxis_showscale=False, xaxis_title="Importance", yaxis_title="")
fig.show()

print("Top feature:", metrics["top_feature"])
print("Second feature:", metrics["second_feature"])
print("Category importance share:", round(metrics["category_importance_share"] * 100, 2), "%")

## Interpretation

The MVP sample contains 160 markets and 40,678 training rows after filtering. The headline pattern is that category matters unevenly rather than uniformly. Relative to US Elections, the clearest category effect is C(category, Treatment(reference='US Elections'))[T.Crypto Corporate] with a higher forward-volatility coefficient of 0.0085 (p=1.05e-195). 6 category term(s) are significant at the 5% level. The OLS adjusted R² is 0.686, so the controls explain a meaningful share of the variation even in a rough one-day build.

The volatility story itself is intuitive. Crypto Airdrops has the highest median realized 7-day volatility (0.0317), while US Elections has the lowest (0.0004). Recent volatility remains a strong positive predictor of future volatility (0.276), but the random forest indicates that bid-ask spread and volume carry most of the non-linear predictive weight, with category contributing about 1.1% of total feature importance. The important caveat is data coverage: the live Polymarket open-market sample is concentrated in a handful of long-duration categories, especially Crypto Airdrops (7 markets), Crypto Corporate (5 markets), Other (3 markets), Hockey (3 markets), and historical spread is approximated with the latest observed bid/ask spread because the ingestion path does not expose a clean daily spread history.